# ⚡ Guaranteed Baseline QLoRA Training on Google Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RYuK006/Research_Paper_Based_Model/blob/main/notebooks/train_research_lm.ipynb)

> **Model Target**: `Qwen2.5-7B-Instruct` 4-bit (with 3B fallback) + LoRA $r=16, \alpha=32$
> **Hardware Budget**: Free Colab T4 (16 GB VRAM, ~12 GB RAM, ~100 GB ephemeral disk) + 5 TB Google Drive
> **Fault Tolerance**: Automatic resumption from Google Drive `manifest.json` on Colab disconnect
> **Evaluation**: Pre-cutoff training vs. post-cutoff held-out perplexity & 10 historical discovery completions

---

### Step 1: Mount Google Drive
Google Drive stores our authoritative checkpoints, tokenized shards (1-5 GB each), 4-bit base model snapshot, and `manifest.json`.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print(f'Drive mount notice: {e}')

### Step 2: Verify GPU & Ephemeral Local Disk
All active training occurs strictly on fast ephemeral local disk (`/content/local_training`).

In [ ]:
!nvidia-smi
!df -h /content

### Step 3: Install High-Performance Unsloth / PEFT Libraries

In [ ]:
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

### Step 4: Clone Research Codebase & Prepare Environment

In [ ]:
import os
if not os.path.exists('Research_Paper_Based_Model'):
    !git clone https://github.com/RYuK006/Research_Paper_Based_Model.git
%cd Research_Paper_Based_Model

### Step 5: Execute Resilient Training Loop (`colab/train.py`)
- Streams shards one-by-one to fast ephemeral disk.
- Checkpoints written locally, verified on Drive, then `manifest.json` is updated.
- Re-running 'Run all' after a disconnect automatically detects the latest checkpoint and resumes seamlessly.

In [ ]:
!python colab/train.py \
    --model-id "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" \
    --fallback-model-id "unsloth/Qwen2.5-3B-Instruct-bnb-4bit" \
    --drive-root "/content/drive/MyDrive/Model_Training" \
    --local-root "/content/local_training" \
    --db-path "data/papers.db" \
    --cutoff-year 2004 \
    --max-steps 1000 \
    --checkpoint-interval 50

### Step 6: Held-Out Evaluation on Post-Cutoff Literature
- Computes perplexity reduction on held-out post-cutoff papers.
- Completes 10 historical discovery prefixes (e.g., $(La,Y)H_{10}$ at 160 GPa).

In [ ]:
import json
eval_path = '/content/local_training/held_out_eval_results.json'
if os.path.exists(eval_path):
    with open(eval_path, 'r', encoding='utf-8') as f:
        results = json.load(f)
    print(f'Base Model Perplexity:      {results["base_perplexity"]:.2f}')
    print(f'Fine-Tuned Perplexity:     {results["finetuned_perplexity"]:.2f} (-{results["perplexity_reduction_pct"]}%)')
    print('\nTop Discovery Completions:')
    for c in results["completions"][:3]:
        print(f'Prompt: {c["prompt"]}')
        print(f'Generated: {c["completion"]}\n')
else:
    print('Evaluation results not found yet. Run Step 5 first.')